## 第 2 節｜隨機森林、SVM、交叉驗證與 Grid Search

### 資料集欄位｜500 封模擬校園郵件

| 欄位 | 用途 |
|---|---|
| `email_id` | 郵件編號，不拿來訓練 |
| `link_count` | 網址連結數量，輸入 X |
| `urgent_words` | 緊急用語次數，輸入 X |
| `exclamation_count` | 驚嘆號次數，輸入 X |
| `sender_known` | 是否為已知寄件者（1 是、0 否），輸入 X |
| `email_type` | 郵件類型，透過 One-Hot 變成數值欄位 |
| `is_spam` | 是否為垃圾郵件（1 是、0 否），答案 y |

先將 `02_校園郵件分類_500筆.csv` 放在 Notebook 的工作目錄。第一個 Code Cell **直接執行**。


In [ ]:
# Import pandas to read the dataset.
# 匯入 pandas，讀取資料集。
import pandas as pd

# Prepare data splits and cross-validation searches.
# 匯入資料切分、交叉驗證與網格搜尋工具。
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Import the random forest used in the main exercise.
# 匯入本節主要實作的隨機森林模型。
from sklearn.ensemble import RandomForestClassifier

# Import the decision tree for model comparisons.
# 匯入決策樹，供選做的模型比較使用。
from sklearn.tree import DecisionTreeClassifier

# Import the SVM introduced in the second lesson slides.
# 匯入第二節投影片介紹的 SVM 模型。
from sklearn.svm import SVC

# Calculate classification accuracy.
# 計算模型分類正確率。
from sklearn.metrics import accuracy_score

# Read the simulated campus email dataset.
# 讀取模擬校園郵件資料集。
df = pd.read_csv('02_校園郵件分類_500筆.csv')

# Preview five email records.
# 預覽五筆郵件資料。
display(df.head())

### 練習 1｜準備 X、y 與測試集（照範例輸入下一格）

```python
# Select only the features used to predict email categories.
# 直接選取要讓模型學習的五個特徵欄位。
features = ['link_count', 'urgent_words', 'exclamation_count', 'sender_known', 'email_type']

# Convert only the categorical email-type column.
# 只將類別型的郵件類型欄位做 One-Hot Encoding。
X = pd.get_dummies(df[features], columns=['email_type'], dtype=int)

# Keep spam/not-spam as the correct answer.
# 將是否為垃圾郵件保留為答案。
y = df['is_spam']

# Reserve 20% of the emails as unseen test data.
# 留下兩成郵件，最後才用來測試模型。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Check the number of emails in each group.
# 確認訓練與測試資料的筆數。
print('訓練:', len(X_train), '測試:', len(X_test))
```

In [ ]:
# 在這格輸入上方「練習 1」的程式碼。


### 練習 2｜訓練隨機森林並測試（照範例輸入下一格）

```python
# Create a forest of 60 small decision trees.
# 建立包含六十棵小樹的隨機森林。
model = RandomForestClassifier(n_estimators=60, max_depth=5, random_state=42)

# Let the forest learn from the training emails.
# 使用訓練資料讓模型學習。
model.fit(X_train, y_train)

# Predict the 100 unseen emails.
# 預測一百封測試郵件。
pred = model.predict(X_test)

# Show the test accuracy and number of mistakes.
# 查看測試正確率及猜錯幾封。
print('Accuracy:', round(accuracy_score(y_test, pred) * 100), '%')

# Count all test mistakes.
# 計算測試資料猜錯幾封。
print('猜錯:', int((pred != y_test).sum()), '封')
```

In [ ]:
# 在這格輸入上方「練習 2」的程式碼。


### Optional 1｜比較決策樹、隨機森林與 SVM，再做三折驗證（直接執行）

對應投影片第 8～10 頁：在同一份測試集比較模型，並使用訓練集進行交叉驗證。

In [ ]:
# Create the decision tree used in the first lesson.
# 建立第一節介紹過的決策樹模型。
tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Train the decision tree on the training emails.
# 使用訓練郵件訓練決策樹。
tree.fit(X_train, y_train)

# Create an SVM classifier for comparison.
# 建立第二節介紹的 SVM 分類模型。
svm = SVC(kernel='rbf')

# Train SVM using the same training emails.
# 使用相同的訓練郵件訓練 SVM。
svm.fit(X_train, y_train)

# Compare the three models on the same held-out test set.
# 在同一份測試集比較三種模型的正確率。
for name, classifier in [('決策樹', tree), ('隨機森林', model), ('SVM', svm)]:
    # Report one comparable accuracy score per model.
    # 顯示各模型在同一份測試資料上的正確率。
    print(name + ':', round(classifier.score(X_test, y_test) * 100), '%')

# Check the forest with three-fold validation on Train only.
# 只使用訓練資料，進行隨機森林的三折交叉驗證。
scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')

# Show the average of the three validation scores.
# 顯示三次驗證正確率的平均值。
print('隨機森林三折平均:', round(scores.mean() * 100, 1), '%')

### Optional 2｜用 Grid Search 尋找較佳的超參數（直接執行）


In [ ]:
# Specify the forest settings to compare.
# 設定要比較的樹木數量與最大深度。
params = {'n_estimators': [20, 60, 100], 'max_depth': [2, 5, 8]}

# Search nine parameter combinations with three-fold validation on Train.
# 在 Train 上使用三折驗證，比較九組超參數。
search = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3, scoring='accuracy')

# Run the grid search without using the held-out test set.
# 執行網格搜尋，不使用保留的測試集挑選設定。
search.fit(X_train, y_train)

# Display the best settings and their validation score.
# 顯示驗證表現較佳的設定及平均正確率。
print('最佳參數:', search.best_params_, '三折平均:', round(search.best_score_ * 100, 1), '%')

# Evaluate the selected model once on the reserved test set.
# 最後才使用保留的測試集驗收選出的模型。
print('Test Accuracy:', round(search.score(X_test, y_test) * 100, 1), '%')
